# DDPM/DDIM with MSRF UNet — Exploration Notebook

This notebook lets you:
- Visualize the forward diffusion process (noise schedule)
- Load a trained model and generate samples
- Compare DDPM vs DDIM sampling
- Sweep CFG scale and eta
- Visualize MSRF gate weights


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from torchvision.utils import make_grid

from models.unet import UNet
from models.diffusion import GaussianDiffusion
from data.datasets import build_dataloaders, denormalize
from utils.training import EMA, load_checkpoint

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Visualize the noise schedule

In [ ]:
diffusion = GaussianDiffusion(timesteps=1000, schedule='cosine')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t = np.arange(1000)

axes[0].plot(t, diffusion.betas.numpy())
axes[0].set_title('β_t (noise added per step)')
axes[0].set_xlabel('Timestep t')

axes[1].plot(t, diffusion.alphas_cumprod.numpy())
axes[1].set_title('ᾱ_t (signal retention)')
axes[1].set_xlabel('Timestep t')

axes[2].plot(t, diffusion.sqrt_alphas_cumprod.numpy(), label='signal', color='blue')
axes[2].plot(t, diffusion.sqrt_one_minus_alphas_cumprod.numpy(), label='noise', color='red')
axes[2].legend()
axes[2].set_title('Signal vs Noise std dev')
axes[2].set_xlabel('Timestep t')

plt.tight_layout()
plt.savefig('noise_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Forward process visualization

In [ ]:
# Load a single CIFAR-10 image and show noising at different timesteps
train_loader, _, _ = build_dataloaders(
    'cifar10', '../data', image_size=32, batch_size=1, num_workers=0, augment=False
)
x0, label = next(iter(train_loader))

timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(16, 2))

for ax, t_val in zip(axes, timesteps_to_show):
    t = torch.tensor([t_val])
    xt = diffusion.q_sample(x0, t)
    img = denormalize(xt[0]).permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(f't={t_val}', fontsize=9)
    ax.axis('off')

plt.suptitle('Forward Diffusion Process (Cosine Schedule)', fontsize=12)
plt.tight_layout()
plt.savefig('forward_process.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Load trained model and generate samples

In [ ]:
# ---- CONFIG (adjust to match your training config) ----
CHECKPOINT = '../outputs/cifar10_cond/ckpts/best.pt'
NUM_CLASSES = 10
IMAGE_SIZE = 32
BASE_CHANNELS = 128
CHANNEL_MULTS = (1, 2, 2, 2)
USE_MSRF = True
NULL_CLASS = NUM_CLASSES
# -------------------------------------------------------

model = UNet(
    image_size=IMAGE_SIZE,
    in_channels=3,
    base_channels=BASE_CHANNELS,
    channel_mults=CHANNEL_MULTS,
    num_res_blocks=2,
    attn_resolutions=(16, 8),
    num_classes=NUM_CLASSES,
    use_msrf=USE_MSRF,
).to(device)

ema = EMA(model)
load_checkpoint(CHECKPOINT, model, ema=ema, device=device)
ema.apply_shadow(model)
model.eval()
print('Model loaded successfully!')

In [ ]:
# Generate one image per CIFAR-10 class with CFG
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

with torch.no_grad():
    labels = torch.arange(10, device=device).repeat(2)  # 2 per class
    samples = diffusion.ddim_sample_loop(
        model, (20, 3, IMAGE_SIZE, IMAGE_SIZE), device,
        ddim_steps=50,
        class_labels=labels,
        cfg_scale=3.0,
        null_class=NULL_CLASS,
        eta=0.0,
    )

grid = make_grid(denormalize(samples), nrow=10)
plt.figure(figsize=(16, 4))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis('off')
plt.title('Class-conditional generation (CFG scale=3.0, DDIM 50 steps)', fontsize=13)
for i, cls in enumerate(CIFAR10_CLASSES):
    plt.text(16 + i * 36, -6, cls, ha='center', fontsize=7, rotation=45)
plt.savefig('conditional_samples.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. CFG Scale Sweep

In [ ]:
# Fix noise, vary CFG scale to visualize guidance effect
cfg_scales = [1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
target_class = 5  # dog

rows = []
for w in cfg_scales:
    torch.manual_seed(42)
    with torch.no_grad():
        labels = torch.full((8,), target_class, device=device)
        s = diffusion.ddim_sample_loop(
            model, (8, 3, IMAGE_SIZE, IMAGE_SIZE), device,
            ddim_steps=50, class_labels=labels,
            cfg_scale=w, null_class=NULL_CLASS, eta=0.0, progress=False,
        )
    rows.append(s)

all_samples = torch.cat(rows, dim=0)
grid = make_grid(denormalize(all_samples), nrow=8)

fig, ax = plt.subplots(figsize=(18, 8))
ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
ax.set_yticks([18 + i * 36 for i in range(len(cfg_scales))])
ax.set_yticklabels([f'w={w}' for w in cfg_scales], fontsize=11)
ax.set_xticks([])
ax.set_title(f'CFG Scale Sweep — Class: {CIFAR10_CLASSES[target_class]}', fontsize=13)
plt.savefig('cfg_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Higher w → more class-faithful but less diverse')

## 5. DDPM vs DDIM comparison

In [ ]:
import time

labels = torch.arange(8, device=device) % 10

# DDPM (full 1000 steps)
torch.manual_seed(0)
t0 = time.time()
with torch.no_grad():
    ddpm_samples = diffusion.p_sample_loop(
        model, (8, 3, IMAGE_SIZE, IMAGE_SIZE), device,
        class_labels=labels, cfg_scale=3.0, null_class=NULL_CLASS,
    )
ddpm_time = time.time() - t0

# DDIM (50 steps)
torch.manual_seed(0)
t0 = time.time()
with torch.no_grad():
    ddim_samples = diffusion.ddim_sample_loop(
        model, (8, 3, IMAGE_SIZE, IMAGE_SIZE), device,
        ddim_steps=50, class_labels=labels,
        cfg_scale=3.0, null_class=NULL_CLASS, eta=0.0,
    )
ddim_time = time.time() - t0

fig, axes = plt.subplots(2, 1, figsize=(16, 5))
for ax, s, title, t in [
    (axes[0], ddpm_samples, f'DDPM (1000 steps, {ddpm_time:.1f}s)', ddpm_time),
    (axes[1], ddim_samples, f'DDIM (50 steps, {ddim_time:.1f}s, {ddpm_time/ddim_time:.1f}x faster)', ddim_time),
]:
    grid = make_grid(denormalize(s), nrow=8)
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('ddpm_vs_ddim.png', dpi=150, bbox_inches='tight')
plt.show()